# AethyxLM - Production Colab Training (T4 GPU)

**Architecture:** 14M params, 8L, 256D, 8H, 128ctx, 32k vocab
**Dataset:** TinyStories (local corpus.txt)
**Storage:** GitHub = code, Google Drive = checkpoints/logs, Colab = compute

---

In [ ]:
# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE (persistent storage)
# ============================================================
from google.colab import drive
import os, sys, subprocess, shutil, json, time, glob

print('Mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)

# Persistent directories on Drive
DRIVE_ROOT = '/content/drive/MyDrive/AethyxLM'
DRIVE_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints')
DRIVE_LOGS = os.path.join(DRIVE_ROOT, 'logs')
DRIVE_TOK = os.path.join(DRIVE_ROOT, 'tokenizer')
DRIVE_DATA = os.path.join(DRIVE_ROOT, 'dataset')

for d in [DRIVE_CKPT, DRIVE_LOGS, DRIVE_TOK, DRIVE_DATA]:
    os.makedirs(d, exist_ok=True)

print(f'[OK] Drive mounted: {DRIVE_ROOT}')
print(f'[OK] Checkpoints: {DRIVE_CKPT}')
print(f'[OK] Logs: {DRIVE_LOGS}')

# ============================================================
# STEP 2: CLONE/PULL FROM GITHUB (code lives in Git)
# ============================================================
REPO_URL = 'https://github.com/aethyx-ai/AethyxLM.git'
LOCAL_ROOT = '/content/AethyxLM'

if os.path.exists(os.path.join(LOCAL_ROOT, '.git')):
    print('Updating existing repo...')
    subprocess.run(['git', '-C', LOCAL_ROOT, 'pull'], check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, LOCAL_ROOT], check=True)

os.chdir(LOCAL_ROOT)
sys.path.insert(0, LOCAL_ROOT)

print(f'[OK] Project: {LOCAL_ROOT}')
print(f'[OK] Config: {os.path.exists("configs/train_config.json")}')
print(f'[OK] Corpus: {os.path.exists("tokenizer/data/corpus.txt")}')

# ============================================================
# STEP 3: INSTALL DEPS
# ============================================================
!pip install tokenizers datasets -q

In [ ]:
# ============================================================
# STEP 4: VERIFY CUDA (T4 on Colab Free)
# ============================================================
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {device_name} ({vram_gb:.1f} GB)')
    assert 'T4' in device_name or 'A100' in device_name or 'V100' in device_name
else:
    raise RuntimeError('CUDA not available! Enable GPU: Runtime → Change runtime type → GPU')

In [ ]:
# ============================================================
# STEP 5: PREPARE DATA FROM LOCAL corpus.txt
# ============================================================
import random

corpus_path = 'tokenizer/data/corpus.txt'
assert os.path.exists(corpus_path), f'Missing {corpus_path}'

with open(corpus_path, 'r', encoding='utf-8') as f:
    content = f.read()

stories = [s.strip() for s in content.split('\n\n') if s.strip()]
print(f'Loaded {len(stories)} stories')

random.seed(42)
random.shuffle(stories)
split = int(0.9 * len(stories))
train_stories = stories[:split]
val_stories = stories[split:]

os.makedirs('data', exist_ok=True)
with open('data/train.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(train_stories))
with open('data/val.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(val_stories))

print(f'Train: {len(train_stories)} | Val: {len(val_stories)}')

# Also copy to Drive for persistence
shutil.copy('data/train.txt', os.path.join(DRIVE_DATA, 'train.txt'))
shutil.copy('data/val.txt', os.path.join(DRIVE_DATA, 'val.txt'))

In [ ]:
# ============================================================
# STEP 6: TRAIN BPE TOKENIZER (32k vocab)
# ============================================================
import subprocess

print('Training tokenizer...')
result = subprocess.run(
    [sys.executable, '-m', 'tokenizer.train_tokenizer'],
    cwd=LOCAL_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Tokenizer training failed')

# Verify
sys.path.insert(0, LOCAL_ROOT)
from tokenizer.tokenizer import AethyxTokenizer
tok = AethyxTokenizer()
print(f'[OK] Vocab size: {tok.vocab_size}')
ids = tok.encode('Hello world')
print(f'[OK] Encode: {ids}')
print(f'[OK] Decode: {tok.decode(ids)}')

# Copy tokenizer to Drive
shutil.copy('tokenizer/tokenizer.json', os.path.join(DRIVE_TOK, 'tokenizer.json'))
shutil.copy('tokenizer/tokenizer.json', os.path.join(DRIVE_TOK, 'tokenizer.json'))
print('[OK] Tokenizer saved to Drive')

In [ ]:
# ============================================================
# STEP 7: CONFIGURE FOR COLAB T4 & AUTO-RESUME
# ============================================================
import json

with open('configs/train_config.json') as f:
    cfg = json.load(f)

# T4-optimized + frequent checkpointing
cfg['training'].update({
    'max_steps': 20000,
    'warmup_steps': 2000,
    'batch_size': 32,
    'grad_accum_steps': 1,
    'use_amp': True,
    'eval_interval': 1000,
    'save_interval': 500,          # Save every 500 steps
    'log_interval': 100,
    'learning_rate': 3e-4,
    'grad_clip': 1.0,
    'weight_decay': 0.1,
    'min_lr_ratio': 0.1,
})

with open('configs/train_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

# Backup config to Drive
shutil.copy('configs/train_config.json', os.path.join(DRIVE_ROOT, 'train_config.json'))

print('[OK] Config updated:')
for k, v in cfg['training'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# STEP 8: AUTO-RESUME FROM DRIVE CHECKPOINT
# ============================================================
def find_latest_checkpoint():
    """Find latest checkpoint in Drive or local."""
    # Priority: Drive latest > local latest > Drive step > local step
    candidates = [
        os.path.join(DRIVE_CKPT, 'checkpoint_latest.pt'),
        'checkpoints/checkpoint_latest.pt',
    ]
    # Add step checkpoints
    for base in [DRIVE_CKPT, 'checkpoints']:
        if os.path.exists(base):
            steps = sorted(glob.glob(os.path.join(base, 'checkpoint_step_*.pt')))
            if steps:
                candidates.append(steps[-1])

    for c in candidates:
        if os.path.exists(c):
            return c
    return None

resume_path = find_latest_checkpoint()
if resume_path:
    print(f'[OK] Found checkpoint: {resume_path}')
    RESUME_FLAG = f'--resume {resume_path}'
else:
    print('[OK] No checkpoint found, starting fresh')
    RESUME_FLAG = ''

In [ ]:
# ============================================================
# STEP 9: SYNC CHECKPOINTS LOCAL ↔ DRIVE
# ============================================================
def sync_checkpoints_to_drive():
    """Copy local checkpoints to Drive."""
    if not os.path.exists('checkpoints'):
        return
    for f in os.listdir('checkpoints'):
        if f.endswith('.pt'):
            src = os.path.join('checkpoints', f)
            dst = os.path.join(DRIVE_CKPT, f)
            try:
                shutil.copy2(src, dst)
            except Exception as e:
                print(f'  Sync failed for {f}: {e}')

def sync_checkpoints_from_drive():
    """Copy Drive checkpoints to local before training."""
    if not os.path.exists(DRIVE_CKPT):
        return
    os.makedirs('checkpoints', exist_ok=True)
    for f in os.listdir(DRIVE_CKPT):
        if f.endswith('.pt'):
            src = os.path.join(DRIVE_CKPT, f)
            dst = os.path.join('checkpoints', f)
            if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                try:
                    shutil.copy2(src, dst)
                    print(f'  Synced from Drive: {f}')
                except Exception as e:
                    print(f'  Sync failed for {f}: {e}')

# Initial sync from Drive
sync_checkpoints_from_drive()
print('[OK] Checkpoint sync ready')

In [ ]:
# ============================================================
# STEP 10: TRAINING WRAPPER WITH AUTO-SYNC
# ============================================================
import torch

print('Starting training on', torch.cuda.get_device_name(0))
print('=' * 60)

cmd = [sys.executable, 'train.py',
       '--config', 'configs/train_config.json',
       '--device', 'cuda']

if RESUME_FLAG:
    cmd.append(RESUME_FLAG)

print(f'Command: {" ".join(cmd)}')
print('-' * 60)

# Run training with periodic sync
import threading, time

stop_sync = False

def periodic_sync():
    while not stop_sync:
        time.sleep(300)  # Every 5 minutes
        if not stop_sync:
            sync_checkpoints_to_drive()
            print(f'[{time.strftime("%H:%M:%S")}] Checkpoints synced to Drive')

sync_thread = threading.Thread(target=periodic_sync, daemon=True)
sync_thread.start()

start = time.time()
result = subprocess.run(cmd, cwd=LOCAL_ROOT)
elapsed = time.time() - start

stop_sync = True
sync_thread.join(timeout=5)

# Final sync
sync_checkpoints_to_drive()

print('=' * 60)
print(f'Training finished in {elapsed/3600:.1f}h')
print(f'Exit code: {result.returncode}')

if result.returncode == 0:
    print('✅ Training completed successfully!')
else:
    print(f'❌ Training failed with code {result.returncode}')
    print('💡 You can resume from last checkpoint on next session')

In [ ]:
# ============================================================
# STEP 11: DOWNLOAD FINAL CHECKPOINTS & LOGS
# ============================================================
from google.colab import files

ckpt_best = 'checkpoints/checkpoint_best.pt'
ckpt_latest = 'checkpoints/checkpoint_latest.pt'
ckpt_steps = sorted([f for f in os.listdir('checkpoints') if f.startswith('checkpoint_step_')])

for f in [ckpt_best, ckpt_latest] + ckpt_steps:
    if os.path.exists(f):
        print(f'Downloading: {f}')
        files.download(f)
    else:
        print(f'Not found: {f}')

# Also copy logs if they exist
if os.path.exists('logs'):
    for f in os.listdir('logs'):
        files.download(os.path.join('logs', f))

In [ ]:
# ============================================================
# STEP 12: QUICK INFERENCE TEST
# ============================================================
import torch
from model.gpt import GPT
from tokenizer.tokenizer import AethyxTokenizer

device = 'cuda'
model = GPT().to(device)
tok = AethyxTokenizer()

ckpt_path = 'checkpoints/checkpoint_best.pt'
if not os.path.exists(ckpt_path):
    ckpt_path = 'checkpoints/checkpoint_latest.pt'

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

@torch.no_grad()
def generate(prompt, max_new=200, temp=0.8, top_k=50):
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new):
        logits = model(ids[:, -128:])
        logits = logits[:, -1, :] / temp
        if top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('inf')
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        ids = torch.cat([ids, next_id], dim=1)
    return tok.decode(ids[0].tolist())

print('Sample generation:')
print('-' * 60)
print(generate('Once upon a time'))
print('-' * 60)
print(generate('The little boy'))
print('-' * 60)
print(generate('In a magical forest'))